In [1]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

/data5/home/anushkapa/.local/lib/python3.8/site-packages/torchvision/io/image.py:11: UserWarning: Failed to load image Python extension: /data5/home/anushkapa/.local/lib/python3.8/site-packages/torchvision/image.so: undefined symbol: _ZNK3c1010TensorImpl36is_contiguous_nondefault_policy_implENS_12MemoryFormatE
  warn(f"Failed to load image Python extension: {e}")


In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '6'

In [3]:
def detectionModel(numClasses):
    ## Load a model pretrained resnet model to speed training time
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
    
    ## Get number of input features for the classifier
    inFeatures = model.roi_heads.box_predictor.cls_score.in_features
    ## Replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(inFeatures, numClasses) 

    return model

In [4]:
model = detectionModel(1)

In [10]:
def load_model(model, folder, net_basename, name):
    state = torch.load(f"{folder}{net_basename}_{name}_model.pkl")
    #state = torch.load(f"{folder}{net_basename}")
    #print(state.keys())
    filtered_state_dict = {k: v for k, v in state['model_rep'].items() if k in model.state_dict()}

# Load the filtered state dictionary into the model
    num_classes = 16  # Set this to the number of classes in your dataset
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)
    model.load_state_dict(filtered_state_dict, strict=False)
    return model

In [6]:
folder = "/data6/anushkapa_scratch/unitary-scalarization-dmtl/vindr/saved_models/saved_models/"
net_basename = "vindr_0-lr:0.01-wd:0.0005_spring-donkey-30"
name = "last"

In [12]:
model = load_model(model, folder, net_basename, name)

In [ ]:
model.eval()

In [ ]:
def init(bboxfile, tasks):
   
    ## Image data
    # dataDF = pd.read_csv(path + dataCSV)
    ## Box data
    boxDF_orig = pd.read_csv(bboxfile)

    boxDF = pd.DataFrame()
    for index, row in boxDF_orig.iterrows():
        row_new = {}
        row_new['patientId'] = row['image_id']
        row_new['x'] = row['x_min']
        row_new['y'] = row['y_min']
        row_new['width'] = row['x_max'] - row['x_min']
        row_new['height'] = row['y_max'] - row['y_min']
        row_new['Target'] = row['class_id']
        boxDF = pd.concat([boxDF, pd.DataFrame([row_new])], ignore_index=True)
    print(boxDF.head())

    boxDF = boxDF[boxDF['Target'].isin(tasks)]
    boxDF.dropna(axis = 0, inplace = True)

    ## Bounding box and label data groupings by image. Using boxDF due to inner join duplication
    xBox = boxDF.groupby('patientId')['x'].apply(np.array).reset_index()['x'].values
    yBox = boxDF.groupby('patientId')['y'].apply(np.array).reset_index()['y'].values
    wBox = boxDF.groupby('patientId')['width'].apply(np.array).reset_index()['width'].values
    hBox = boxDF.groupby('patientId')['height'].apply(np.array).reset_index()['height'].values
    ## Group the finding labels together for the varying bounding boxes in each image
    boxLabel = boxDF.groupby('patientId')['Target'].apply(np.array).reset_index()

    print(xBox)

    print(boxLabel.head())

    boxLabel['paths'] = os.path.join(path,imageFolders[0]) + boxLabel['patientId'] + ".png"

    print("Number of images: ", len(boxLabel))
    print(boxLabel.head())
    return xBox, yBox, wBox, hBox, boxLabel

In [ ]:
from lung_data import LungData
height = 256
width = 256
wh_pathDir = "/data6/rajivporana_scratch/vindr_data/wh_proper_test/"
lung_segment_path = "/data6/rajivporana_scratch/vindr_bbox/dataset/png_test_lung_segment/"
target_dir = "/data6/anushkapa_scratch/unitary-scalarization-dmtl/vindr/saved_results/"
dirname = "/data6/anushkapa_scratch/unitary-scalarization-dmtl/vindr/"
# train_csv = "vindr_train.csv"
val_csv = "vindr_val.csv"

print(args.tasks)
tasks = [int(x) for x in args.tasks]
print("tasks : ", tasks)

xBox, yBox, wBox, hBox, boxLabel = init(os.path.join(dirname, val_csv), tasks)
pngImages = create_png_df(boxLabel)
dataTest = LungData(height, width, xBox, yBox, wBox, hBox, boxLabel, 
    pngImages, classes, global_transformer(), wpPath=wh_pathDir)

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=4, collate_fn=collate_fn)